# Phase 2: Hybrid Model Training Pipeline
This notebook contains the training logic for the hybrid weapon detector.

### 1. Imports

In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from models.hybrid_model import HybridWeaponDetector
import os

### 2. Dataset Definition

In [ ]:
class DummyYOLODataset(Dataset):
    """Dummy dataset for verification purposes."""
    def __init__(self, size=100):
        self.size = size
        
    def __len__(self):
        return self.size
        
    def __getitem__(self, idx):
        # Return a dummy image (C, H, W) and dummy targets
        # Targets shape would normally depend on the specific loss function requirements
        img = torch.randn(3, 640, 640)
        
        # We need targets for classification, box regression, and objectness
        # For this dummy test, we just return zeros of arbitrary shapes
        # In a real scenario, this would parse YOLO txt labels
        return img, torch.zeros(0)

### 3. Training Loop

In [ ]:
def train(model, dataloader, epochs=10, device="cuda"):
    model.to(device)
    model.train()
    
    # Freeze backbone initially
    model.backbone.freeze()
    print("Backbone frozen for initial epochs.")
    
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
    
    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs}")
        
        # Unfreeze backbone after some epochs (e.g., epoch 5)
        if epoch == 5:
            model.backbone.unfreeze()
            # Update optimizer with new parameters
            optimizer = optim.AdamW(model.parameters(), lr=1e-5)
            print("Backbone unfrozen.")
            
        epoch_loss = 0.0
        
        for imgs, targets in tqdm(dataloader):
            imgs = imgs.to(device)
            # targets = targets.to(device)
            
            optimizer.zero_grad()
            
            # Forward pass
            cls_logits, bbox_offsets, objectness = model(imgs)
            
            # Dummy loss computation (just to verify graph connectivity)
            # In reality, you'd match anchors, compute focal loss and DFL
            dummy_loss = cls_logits.sum() * 0.0 + bbox_offsets.sum() * 0.0 + objectness.sum() * 0.0
            dummy_loss += torch.tensor(1.0, requires_grad=True).to(device) # prevent graph error
            
            dummy_loss.backward()
            optimizer.step()
            
            epoch_loss += dummy_loss.item()
            
        print(f"Loss: {epoch_loss/len(dataloader)}")
        
    # Save model
    os.makedirs("models/weights", exist_ok=True)
    model.save("models/weights/best.pt")
    print("Training complete. Model saved.")

### 4. Execution

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Instantiate model
model = HybridWeaponDetector(backbone_variant="yolo11n.pt", pretrained=True, nc=3, device=device)

# Dataloader
dataset = DummyYOLODataset(size=10)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# Train
train(model, dataloader, epochs=6, device=device)